# COMP560 ReID — Kaggle Runner

Kaggle has no Google Drive. We pull datasets via `gdown` (share-link file IDs), train, then upload checkpoints back to a Drive folder via a short-lived service-account (or just download the notebook output).

Fill in the **Config** cell, then Run All. Turn on GPU (T4 / P100).

In [ ]:
# ====== Config ======
CONFIG_NAME   = 'exp03_baseline_resnet50_cross'   # change per run
REPO_URL      = 'https://github.com/AlexTourneux/ReID_final.git'  # TODO: fill in
BRANCH        = 'main'

# Google Drive file IDs (get these from sharable links on the shared folder)
DATASET_A_ZIP_ID = 'PASTE_FILE_ID_FOR_images_dataset_a.zip'
MARKET1501_ZIP_ID = 'PASTE_FILE_ID_FOR_market1501.zip'
OUR_TRAIN_ID     = 'PASTE_FILE_ID_FOR_our_train.parquet'
OUR_TEST_ID      = 'PASTE_FILE_ID_FOR_our_test.parquet'
MARKET_TRAIN_ID  = 'PASTE_FILE_ID_FOR_market1501_train.parquet'
MARKET_TEST_ID   = 'PASTE_FILE_ID_FOR_market1501_test.parquet'

# For KD runs — set to a Drive file id of the teacher's best.pth, or leave blank
TEACHER_CKPT_ID  = ''

WORKDIR = '/kaggle/working/object-reid'
print('CONFIG_NAME:', CONFIG_NAME)

In [ ]:
# ====== Clone repo ======
import subprocess, os, shutil
if os.path.isdir(WORKDIR):
    shutil.rmtree(WORKDIR)
subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, WORKDIR])
os.chdir(WORKDIR)
!git log -1 --oneline

In [ ]:
# ====== Install deps ======
!pip install -q -r requirements.txt

In [ ]:
# ====== Download datasets via gdown ======
import os, gdown
os.makedirs('datasets/dataset_a', exist_ok=True)
os.makedirs('datasets/market1501', exist_ok=True)

def gdown_file(fid, out):
    if fid and not os.path.exists(out):
        gdown.download(f'https://drive.google.com/uc?id={fid}', out, quiet=False)

gdown_file(DATASET_A_ZIP_ID,  'datasets/dataset_a/images.zip')
gdown_file(MARKET1501_ZIP_ID, 'datasets/market1501/market1501.zip')
gdown_file(OUR_TRAIN_ID,   'datasets/dataset_a/our_train.parquet')
gdown_file(OUR_TEST_ID,    'datasets/dataset_a/our_test.parquet')
gdown_file(MARKET_TRAIN_ID,'datasets/market1501/market1501_train.parquet')
gdown_file(MARKET_TEST_ID, 'datasets/market1501/market1501_test.parquet')

if not os.path.isdir('datasets/dataset_a/images'):
    !unzip -q datasets/dataset_a/images.zip -d datasets/dataset_a
if not os.path.isdir('datasets/market1501/bounding_box_train'):
    !unzip -q datasets/market1501/market1501.zip -d datasets/market1501

if not os.path.exists('datasets/market1501/market1501_train.parquet'):
    !python scripts/prepare_market1501.py --market_root datasets/market1501

In [ ]:
# ====== (Optional) pull teacher checkpoint for KD runs ======
import os, gdown
if TEACHER_CKPT_ID:
    teacher_dst = 'checkpoints/exp01_teacher_dinov2_large/best.pth'
    os.makedirs(os.path.dirname(teacher_dst), exist_ok=True)
    gdown.download(f'https://drive.google.com/uc?id={TEACHER_CKPT_ID}', teacher_dst, quiet=False)
    print('Teacher loaded:', teacher_dst)

In [ ]:
# ====== Train ======
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!python train.py --config configs/experiments/$CONFIG_NAME.yaml

In [ ]:
# ====== Save outputs to /kaggle/working ======
# Kaggle sessions persist output under /kaggle/working. Download it after
# the run and upload to Drive manually, OR use the 'Save Version' output.
import os, shutil
out_dir = f'/kaggle/working/ckpt_out/{CONFIG_NAME}'
os.makedirs(os.path.dirname(out_dir), exist_ok=True)
if os.path.isdir(out_dir):
    shutil.rmtree(out_dir)
shutil.copytree(f'checkpoints/{CONFIG_NAME}', out_dir)
!ls -lh "$out_dir"